### Structured Output

You can ask models to return responses in a predefined format or structure. This makes the output easier to read, validate, and use in applications or later processing steps. LangChain provides different ways to define schemas and generate structured responses reliably.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
# os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D1978FA900>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D1978FB620>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")
    exits: bool=Field(description="whether the movie exists or not")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D1978FA900>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D1978FB620>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out o

In [4]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. It\'s a 2010 movie directed by Christopher Nolan. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who enters people\'s dreams to steal secrets. The movie\'s title, Inception, refers to the process of planting an idea into someone\'s subconscious. \n\nThe plot involves a team of specialists who use a machine called the PAS to enter dreams. There\'s a concept called "The Dream Within a Dream Within a Dream," where they go deeper into layers of dreams. I think there\'s a character named Arthur who\'s Cobb\'s partner, and a woman named Ariadne who is an architect, designing the dream worlds. There\'s also a dream-sharing device that allows the team to control the environment. \n\nThe main conflict is Cobb trying to get a job offer from a client named Robert Fischer, whose father has a business empire. The team needs to plant th

In [6]:
response=model_with_structure.invoke("Provide details about the moview spiderman")
response

Movie(title='Spiderman', year=2002, director='Sam Raimi', rating=7.5, exits=True)

### Message output with parsed structure

model.with_structured_output(Movie, include_raw=True)  
raw ouput + structured output

In [7]:
# include_raw=True
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

# response = model_with_structure.invoke("Provide details about the movie dhurander 2026")
response = model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception. The title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yep, IMDb gives it 8.8/10. So I have all the required parameters. I\'ll structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: title and director as strings, year as an integer, and rating as a number. No need for any other functions here since the user just wants the movie details. Alright, that should cover it.\n', 'tool_calls': [{'id': '247nns6nw', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","

### Nested Structure

In [18]:
# a movie can have 
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor] # nested structure
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD") # float or default is none
    exists: bool

model_with_structure = model.with_structured_output(MovieDetails)

# response = model_with_structure.invoke("Provide details about the movie 3 idiots")
response = model_with_structure.invoke("Provide details about the movie spiderman")
# response = model_with_structure.invoke("Provide details about the movie name as 2026 ")

# Validation logic
# if not response.cast and not response.genres:
#     print("Movie likely does not exist")

# else:
#     print("Movie exists")

response

MovieDetails(title='Spiderman', year=2002, cast=[Actor(name='Tobey Maguire', role='Peter Parker / Spiderman'), Actor(name='Willem Dafoe', role='Green Goblin'), Actor(name='Kirsten Dunst', role='Mary Jane Watson')], genres=['Action', 'Adventure', 'Superhero'], budget=None, exists=True)

In [22]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

c:\Users\Administrator\Downloads\b150-genaiops\18_Langchain\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='25463b8c-ca4c-4bd3-84af-1d6fc110e027'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'awvedffc3', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 288, 'total_tokens': 319, 'completion_time': 0.035545373, 'completion_tokens_details': None, 'prompt_time': 0.04054699, 'prompt_tokens_details': None, 'queue_time': 0.038000992, 'total_time': 0.076092363}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e128d-c790-7d23-a200-6c84b1e61146-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.
- if in output even if you mentioned str but you got from llm as int then typedict will not show any error
- the output it create is in dict form

In [ ]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, "The title of the movie"] # Annotated[actual_type, metadata1, metadata2, metadata3]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie spiderman")
response

BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool MovieDict did not match schema: errors: [`/title`: expected integer, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<tool_call>\n{"name": "MovieDict", "arguments": {"director": "Sam Raimi", "rating": 7.5, "title": "Spiderman", "year": 2002}}\n</tool_call>'}}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator